# Rare crash narratives: FLAN-T5-base GPU run

Runs `scripts/gpu_run.sh` on a Colab GPU. Choose **Runtime → Change runtime type → GPU**, then **Runtime → Run all**.

Data, checkpoints and results are stored on Google Drive in `MyDrive/coms4705_rarecrash`. If the session disconnects, reconnect and **Run all** again: finished stages are skipped, and an unfinished training run continues from its last evaluation (every 250 steps).

The notebook also runs from VS Code with Google's Colab extension: open it, choose **Select Kernel → Colab**, and pick a GPU server. If the Drive cell reports that Drive is not mounted, run **Colab: Mount Google Drive to Server...** from the Command Palette, run the cell it adds, and continue.

Each session checks out the commit recorded when the experiment first started, so a resumed experiment always runs the same code. To start a fresh experiment on newer code, delete `MyDrive/coms4705_rarecrash/experiment_commit.txt` and the results folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import os
from google.colab import drive

if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except Exception as error:
        print('drive.mount failed:', error)
if not os.path.ismount('/content/drive'):
    raise SystemExit('Google Drive is not mounted, so results would be lost on disconnect. '
                     'In VS Code, run "Colab: Mount Google Drive to Server..." from the Command '
                     'Palette, run the cell it adds, then run this cell again.')
DRIVE_DIR = '/content/drive/MyDrive/coms4705_rarecrash'
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
REPO_URL = 'https://github.com/ncbajaj/COMS_4705_Final_Project.git'
REPO = '/content/COMS_4705_Final_Project'
COMMIT_FILE = f'{DRIVE_DIR}/experiment_commit.txt'

if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
!git -C {REPO} fetch -q origin

if os.path.exists(COMMIT_FILE):
    commit = open(COMMIT_FILE).read().strip()
else:
    commit = !git -C {REPO} rev-parse origin/main
    commit = commit[0].strip()
    open(COMMIT_FILE, 'w').write(commit + '\n')
!git -C {REPO} checkout -q {commit}
print('experiment commit:', commit)

In [ ]:
# Keep data and results on Drive so they survive a disconnect.
for name in ('data', 'runs'):
    target = f'{DRIVE_DIR}/{name}'
    link = f'{REPO}/{name}'
    os.makedirs(target, exist_ok=True)
    if os.path.islink(link):
        continue
    if os.path.exists(link):
        raise RuntimeError(f'{link} exists and is not a link to Drive; remove it and rerun this cell')
    os.symlink(target, link)
!ls -la {REPO} | grep -E ' (data|runs) '

## Settings

The defaults are the full experiment: uniform and frequency-aware sampling, three training seeds each, 3,000 steps at batch size 16. `SCENARIO = '1'` runs the variant whose targets include the collision-type sentence and writes it to `runs/gpu-scenario`, separate from the main results. Any other variable listed at the top of `scripts/gpu_run.sh` can be added here.

In [ ]:
SETTINGS = {
    'SCENARIO': '0',
    'TRAIN_SEEDS': '4705 4706 4707',
    'SAMPLE_SEEDS': '1 2 3',
    'STEPS': '3000',
    'ALPHA': '0.5',
}
OUT = 'runs/gpu-scenario' if SETTINGS['SCENARIO'] == '1' else 'runs/gpu'
ENV = ' '.join(f'{k}="{v}"' for k, v in SETTINGS.items())
print(ENV, '->', OUT)

## Run

Installs the pinned packages from `requirements-train.txt` into Colab's Python, then runs the whole experiment. Output is also appended to `MyDrive/coms4705_rarecrash/gpu_run_log.txt`.

In [ ]:
!cd {REPO} && USE_VENV=0 PYTHON=python3 {ENV} bash scripts/gpu_run.sh 2>&1 | tee -a {DRIVE_DIR}/gpu_run_log.txt

## Results

In [ ]:
from IPython.display import Markdown, display

path = f'{REPO}/{OUT}/comparison.md'
if os.path.exists(path):
    display(Markdown(open(path).read()))
else:
    print('No comparison yet; the run has not finished.')

In [ ]:
# Drive usage: each trained FLAN-T5-base model is about 1 GB, and an unfinished
# run also keeps a resume file of about 3 GB until it completes.
!du -sh {DRIVE_DIR}/runs/* 2>/dev/null
!df -h /content/drive | tail -1